In [2]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

Defaulting to user installation because normal site-packages is not writeable


# Ingestion Pipeline

In [4]:
# Data => Documents
import os
from langchain_community.document_loaders import PyPDFLoader

### Document

In [6]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [7]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 80


In [10]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [12]:
# chunks
!pip install langchain_text_splitters

Defaulting to user installation because normal site-packages is not writeable


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [16]:
chunks = split_docs(all_pdf_documents)

In [24]:
len(chunks)
# type(chunks)

640

### Embedding

In [20]:
from sentence_transformers import SentenceTransformer

In [29]:
class EmbeddingManager:
    # load embedding model and make an instance of it to be used for generating any embedding
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [30]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


In [33]:
# testing embeddings
text = ["Python is a programming language.", "Python supports OOP"]

embedding = embedding_manager.generate_embeddings(text)
# print(embedding)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (2, 384)


### Vector

In [34]:
import chromadb
import uuid

In [35]:
class VectorStoreManager:
    # make a directory for chromaDB 
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None # specify collection only when directory made later
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        # store both original chunked docs and embedded chunks
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
        # to access two data structures at a time in a loop, use zip
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [36]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [37]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

embeddings shape: (640, 384)
total documents added in vector store= 640
docs in collection: 640


# Retrieval Pipeline

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [41]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [49]:
rag_retriever.retrieve("What is neuron-level model")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_a4c10b48-9502-4920-8681-b2e6e2331672',
  'document': '(ignore the ‘1’ dimension here for simplicity).\n𝑀 effectively defines the length of the history of pre-activations that each neuron-level model works\nwith. We tested a range of values for𝑀, and found a range of 10-100 to be effective. Each neuron,\n{1,...,𝐷 }, is then given its own\n3 privately parameterized model that produces what we consider\n4\npost-activations:\nz𝑡+1\n𝑑 = 𝑔𝜃𝑑(A𝑡\n𝑑), (3)\nwhere𝜃𝑑 are the unique parameters for neuron𝑑, andz𝑡+1\n𝑑 is a single unit in the vector that contains',
  'metadata': {'doc_index': 50,
   'total_pages': 59,
   'subject': '',
   'author': '',
   'page': 6,
   'producer': 'pdfTeX-1.40.25',
   'page_label': '7',
   'moddate': '2025-05-29T00:22:55+00:00',
   'creationdate': '2025-05-29T00:22:55+00:00',
   'content_length': 490,
   'trapped': '/False',
   'keywords': '',
   'source': 'data/pdfs\\research1.pdf',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pd

# Integrate with LLMs'

## Anthropic - Claude

In [66]:
!pip install -U langchain-anthropic

Defaulting to user installation because normal site-packages is not writeable


In [56]:
ANTHROPIC_API_KEY="api_key_here"

In [64]:
# create claude LLM
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    api_key = ANTHROPIC_API_KEY,
    model="claude-sonnet-4-5-20250929",
    temperature=0.1,
    max_tokens=1024,
    # timeout=,
    # max_retries=,
    # base_url="...",
    # Refer to API reference for full list of parameters
)

In [65]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    # retrieve information
    results = retriever.retrieve(query, top_k)
    
    # Extract text from retrieved chunks
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    # Case if nothing relevant was retrieved
    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [ ]:
answer = generate_output("what is neuron-level model ?", rag_retriever, llm)

In [ ]:
print(answer)